# Laboratorios de Pandas: limpieza, análisis y combinación de datos (Instacart)

**Actividad:** GA1-220501094-01-AA1-EV06 · Limpieza, análisis y estructuración de datos con Pandas (Instacart)

**Programa:** Análisis y Desarrollo de Software · SENA · Ficha 3223873

**Aprendiz:** Nicolás Muñoz Hernández

**Fecha:** 11 de noviembre de 2026

**Avance de este cuaderno:** Lecciones 2, 3 y 4

**Pregunta orientadora:** ¿cómo preparar y estructurar los datos de Instacart para obtener información confiable sobre los hábitos de compra?

| Parte | Contenido |
|---|---|
| Preparación | Carga de los cinco archivos CSV |
| Lección 2 | Limpieza y estandarización · Laboratorio guiado 2 · Reto 2 |
| Lección 3 | Filtrado, ordenamiento y agrupación · Laboratorio guiado 3 · Reto 3 |
| Lección 4 | Combinación de datasets · Laboratorio guiado 4 · Reto 4 |

## Preparación: carga de los datos

Los cinco archivos están en `data/raw/` (se conservan sin modificar) y usan `;` como separador. Los resultados procesados se guardarán en `data/processed/`.

In [50]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

# El cuaderno vive en notebooks/, así que los datos están un nivel arriba.
RUTA = next((c for c in [Path('../data/raw'), Path('data/raw')] if (c / 'instacart_orders.csv').exists()), None)
assert RUTA is not None, 'No encuentro los CSV: ponlos en data/raw/ (ver README.md).'
RAIZ = RUTA.parent.parent
SALIDA = RAIZ / 'data' / 'processed'
INFORMES = RAIZ / 'reports'
SALIDA.mkdir(parents=True, exist_ok=True)
INFORMES.mkdir(parents=True, exist_ok=True)

fmt = lambda n: f'{n:,.0f}'.replace(',', '.')      # 1234567 -> 1.234.567

try:
    import pyarrow
    print('pyarrow', pyarrow.__version__)
except ImportError:
    print('AVISO: falta pyarrow. Instálalo con  python -m pip install pyarrow  y reinicia el kernel.')
print('pandas', pd.__version__)
print('Datos leídos desde:', RUTA.resolve())

pyarrow 26.0.1
pandas 3.1.0
Datos leídos desde: C:\Users\IK\Documents\instacart-analisis\data\raw


In [51]:
orders = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products = pd.read_csv(RUTA / 'products.csv', sep=';')
aisles = pd.read_csv(RUTA / 'aisles.csv', sep=';')
departments = pd.read_csv(RUTA / 'departments.csv', sep=';')
order_prods = pd.read_csv(RUTA / 'order_products.csv', sep=';')

for nombre, df in {'orders': orders, 'products': products, 'aisles': aisles,
                   'departments': departments, 'order_prods': order_prods}.items():
    print(f'{nombre:12s}', df.shape)
# Deberías ver: (478967, 6) (49694, 4) (134, 2) (21, 2) (4545007, 4)

orders       (478967, 6)
products     (49694, 4)
aisles       (134, 2)
departments  (21, 2)
order_prods  (4545007, 4)


---
# Lección 2 · Limpieza y estandarización de datos crudos

Para cada problema hay tres opciones: **eliminar, corregir o conservar**, y la decisión depende de qué significa el dato. Se sigue siempre este orden: duplicados → tipos → nulos → texto → duplicados implícitos → verificación.

## 2.1 Ejemplos de la lección

### Duplicados exactos: `duplicated` y `drop_duplicates`

In [52]:
duplicados = orders[orders.duplicated(keep=False)].sort_values('order_id')
print(len(duplicados))
duplicados[['order_id', 'order_dow', 'order_hour_of_day']].head(6)
# Deberías ver 30 filas: 15 pedidos y sus 15 copias, todas en order_dow = 3 y hora 2

30


,order_id,order_dow,order_hour_of_day
354993,391768,3,2
371905,391768,3,2
119251,408114,3,2
321100,408114,3,2
394347,467134,3,2
250626,467134,3,2


Todas las copias ocurren el mismo día y a la misma hora (miércoles, 2 a. m.), lo que apunta a un fallo del sistema y no a pedidos reales. **Decisión: eliminar.**

In [53]:
orders = orders.drop_duplicates().reset_index(drop=True)
print(orders.shape, orders['order_id'].is_unique)
# Deberías ver (478952, 6) True

(478952, 6) True


### Tipos de dato con `astype`

Se reducen los enteros al tamaño necesario y se usa `Int16` (con mayúscula, *nullable*) donde hay nulos.

In [54]:
print('Antes:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

orders = orders.astype({
    'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
    'order_dow': 'int8', 'order_hour_of_day': 'int8',
    'days_since_prior_order': 'Int16',
})
order_prods = order_prods.astype({
    'order_id': 'int32', 'product_id': 'int32',
    'add_to_cart_order': 'Int16', 'reordered': 'int8',
})

print('Después:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')
# Deberías ver Antes: 23.0 MB y Después: 7.2 MB (si la ejecutas dos veces, "Antes" ya muestra la tabla optimizada)

Antes: 23.0 MB
Después: 7.2 MB


### Nulos: comprobar la causa antes de decidir

In [55]:
nulos = orders['days_since_prior_order'].isna()
primeros = orders['order_number'] == 1
print((nulos == primeros).all())
# Deberías ver True: cada nulo es el primer pedido de un cliente

True


Cada nulo de `days_since_prior_order` es un **primer pedido** (no existe pedido anterior). **Decisión: conservar el nulo.** Rellenar con 0 inventaría que el cliente volvió el mismo día y bajaría la media de espera.

In [56]:
print(products[products['product_name'].isna()][['aisle_id', 'department_id']].value_counts())
products['product_name'] = products['product_name'].fillna('Unknown')
print(products['product_name'].isna().sum())
# Deberías ver una sola combinación (100, 21) con 1258 productos, y luego 0 nulos

aisle_id  department_id
100       21               1258
Name: count, dtype: int64
0


Los productos sin nombre pertenecen todos al pasillo y departamento `missing`. **Decisión: corregir** con un valor explícito (`'Unknown'`); el producto existe y no se puede borrar.

| Columna | Causa | Decisión |
| --- | --- | --- |
| `days_since_prior_order` | Primer pedido del cliente | Conservar el nulo |
| `product_name` | Producto sin datos de catálogo | Rellenar con `'Unknown'` |
| `add_to_cart_order` | Se investiga en el Reto 2 | Conservar el nulo |

### Texto con los métodos `.str`

In [57]:
nombres = products['product_name']
print('Espacios dobles:', nombres.str.contains('  ', regex=False).sum())
print('Espacio invisible:', nombres.str.contains('\xa0', regex=False).sum())

products['product_name'] = (
    products['product_name']
    .str.replace('\xa0', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
)
print(products['product_name'].str.contains('  ', regex=False).sum(),
      products['product_name'].str.contains('\xa0', regex=False).sum())
# Deberías ver Espacios dobles: 403, Espacio invisible: 16 y, tras limpiar, 0 0

Espacios dobles: 403
Espacio invisible: 16
0 0


### Duplicados implícitos

Aparecen solo después de estandarizar (por ejemplo `American Cheese slices` y `American Cheese Slices`).

In [58]:
conocidos = products[products['product_name'] != 'Unknown']
repetidos = conocidos[conocidos['product_name'].duplicated(keep=False)]
print(conocidos['product_name'].duplicated().sum())
print(repetidos['product_id'].isin(order_prods['product_id']).sum(), 'de', len(repetidos))
# Deberías ver 141 y "268 de 280"

141
268 de 280


Los nombres repetidos tienen `product_id` distintos y casi todos aparecen en pedidos. **Decisión: conservar y documentar.** Borrarlos dejaría pedidos apuntando a productos inexistentes; en los análisis se agrupa por `product_id`, no por nombre.

### Limpieza reutilizable con funciones

In [59]:
def limpiar_orders(df):
    df = df.drop_duplicates().reset_index(drop=True)
    df = df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                    'order_dow': 'int8', 'order_hour_of_day': 'int8',
                    'days_since_prior_order': 'Int16'})
    assert df['order_id'].is_unique
    return df

def limpiar_products(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

print(limpiar_orders(orders).shape)
# Deberías ver (478952, 6): aplicada a una tabla ya limpia, no cambia nada

(478952, 6)


In [60]:
assert orders.shape == (478952, 6)
assert orders['days_since_prior_order'].isna().sum() == 28817   # 2 nulos estaban entre los duplicados
assert products['product_name'].isna().sum() == 0
assert str(order_prods['add_to_cart_order'].dtype) == 'Int16'
print('Limpieza verificada: todas las comprobaciones pasaron')

Limpieza verificada: todas las comprobaciones pasaron


## 2.2 Laboratorio guiado 2 · Pipeline de limpieza con informe de cambios

Se parte de los archivos **crudos** y se encadenan funciones pequeñas con `pipe`; después de cada paso se anota el estado de la tabla para poder responder "¿qué le hiciste a los datos?" con cifras.

### Paso 1: recargar los datos crudos

In [61]:
orders_crudo   = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
op_crudo       = pd.read_csv(RUTA / 'order_products.csv', sep=';')
print(orders_crudo.shape, products_crudo.shape, op_crudo.shape)
# Deberías ver (478967, 6) (49694, 4) (4545007, 4)

(478967, 6) (49694, 4) (4545007, 4)


### Paso 2: una función que anota el estado de la tabla

In [62]:
registro = []

def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df

### Paso 3: un paso de limpieza por función

In [63]:
def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def tipos_orders(df):
    return df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                      'order_dow': 'int8', 'order_hour_of_day': 'int8',
                      'days_since_prior_order': 'Int16'})

def limpiar_nombres(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

def tipos_order_prods(df):
    return df.astype({'order_id': 'int32', 'product_id': 'int32',
                      'add_to_cart_order': 'Int16', 'reordered': 'int8'})

### Paso 4: encadenar los pasos con `pipe`

In [64]:
registro = []   # se reinicia aquí para no duplicar filas si se vuelve a ejecutar

orders_limpio = (orders_crudo
    .pipe(anotar, 'orders', '0 crudo')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'orders', '1 sin duplicados')
    .pipe(tipos_orders)
    .pipe(anotar, 'orders', '2 tipos'))

products_limpio = (products_crudo
    .pipe(anotar, 'products', '0 crudo')
    .pipe(limpiar_nombres)
    .pipe(anotar, 'products', '1 nombres'))

op_limpio = (op_crudo
    .pipe(anotar, 'order_prods', '0 crudo')
    .pipe(tipos_order_prods)
    .pipe(anotar, 'order_prods', '1 tipos'))

### Paso 5: el informe de cambios

In [65]:
informe = pd.DataFrame(registro)
informe

,tabla,paso,filas,nulos,memoria_mb
0,orders,0 crudo,478967,28819,23.0
1,orders,1 sin duplicados,478952,28817,23.0
2,orders,2 tipos,478952,28817,7.2
3,products,0 crudo,49694,1258,3.1
4,products,1 nombres,49694,0,3.1
5,order_prods,0 crudo,4545007,836,145.4
6,order_prods,1 tipos,4545007,836,54.5


In [66]:
ob = informe.set_index(['tabla', 'paso'])
dup_quitados = ob.loc[('orders', '0 crudo'), 'filas'] - ob.loc[('orders', '1 sin duplicados'), 'filas']
nulos_quitados = ob.loc[('orders', '0 crudo'), 'nulos'] - ob.loc[('orders', '1 sin duplicados'), 'nulos']
mem_o0, mem_o2 = ob.loc[('orders', '0 crudo'), 'memoria_mb'], ob.loc[('orders', '2 tipos'), 'memoria_mb']
mem_p0, mem_p1 = ob.loc[('order_prods', '0 crudo'), 'memoria_mb'], ob.loc[('order_prods', '1 tipos'), 'memoria_mb']
nulos_op = ob.loc[('order_prods', '1 tipos'), 'nulos']
nulos_pr0, nulos_pr1 = ob.loc[('products', '0 crudo'), 'nulos'], ob.loc[('products', '1 nombres'), 'nulos']
display(Markdown(
    f"**Interpretación del informe:**\n\n"
    f"- Quitar duplicados eliminó {dup_quitados} filas de `orders` y {nulos_quitados} nulos (copias de pedidos que eran primeros pedidos).\n"
    f"- Corregir tipos no cambió filas ni nulos, pero redujo la memoria de `orders` de {mem_o0} a {mem_o2} MB y la de `order_prods` de {mem_p0} a {mem_p1} MB.\n"
    f"- Limpiar nombres pasó los nulos de `products` de {nulos_pr0} a {nulos_pr1}.\n"
    f"- Los {nulos_op} nulos de `order_prods` (`add_to_cart_order`) siguen ahí a propósito: se decidió conservarlos."
))

**Interpretación del informe:**

- Quitar duplicados eliminó 15 filas de `orders` y 2 nulos (copias de pedidos que eran primeros pedidos).
- Corregir tipos no cambió filas ni nulos, pero redujo la memoria de `orders` de 23.0 a 7.2 MB y la de `order_prods` de 145.4 a 54.5 MB.
- Limpiar nombres pasó los nulos de `products` de 1258 a 0.
- Los 836 nulos de `order_prods` (`add_to_cart_order`) siguen ahí a propósito: se decidió conservarlos.

### Paso 6: validar el resultado

In [67]:
def validar(o, p, op):
    assert o['order_id'].is_unique, 'order_id repetido'
    assert o['order_hour_of_day'].between(0, 23).all(), 'hora fuera de rango'
    assert p['product_name'].notna().all(), 'productos sin nombre'
    assert not p['product_name'].str.contains('\xa0', regex=False).any(), 'espacios invisibles'
    assert op['reordered'].isin([0, 1]).all(), 'reordered con valores raros'
    return 'Validación superada'

print(validar(orders_limpio, products_limpio, op_limpio))
print(orders_limpio.equals(orders), products_limpio.equals(products))
# Deberías ver "Validación superada" y "True True"

Validación superada
True True


El pipeline produce exactamente las mismas tablas que la limpieza manual: la limpieza queda **reproducible** y validada con `assert`.

## 2.3 Reto 2 · Siete ejercicios

### Ejercicio 1 · Duplicados exactos, pares repetidos y valores de `reordered` en `order_prods`

In [68]:
dup_exactos = int(order_prods.duplicated().sum())
dup_pares = int(order_prods.duplicated(subset=['order_id', 'product_id']).sum())
valores_reordered = sorted(order_prods['reordered'].unique().tolist())
print('Filas duplicadas exactas:', dup_exactos)
print('Pares order_id + product_id repetidos:', dup_pares)
print('Valores de reordered:', valores_reordered)

limpio = dup_exactos == 0 and dup_pares == 0 and set(valores_reordered) <= {0, 1}
display(Markdown(
    f"**Interpretación:** hay {fmt(dup_exactos)} duplicados exactos y {fmt(dup_pares)} pares repetidos; `reordered` solo contiene {valores_reordered}. "
    + ("La tabla no necesita limpieza en estos tres puntos." if limpio else "Hay que revisar estos registros antes de analizar.")
))

Filas duplicadas exactas: 0
Pares order_id + product_id repetidos: 0
Valores de reordered: [0, 1]


**Interpretación:** hay 0 duplicados exactos y 0 pares repetidos; `reordered` solo contiene [0, 1]. La tabla no necesita limpieza en estos tres puntos.

### Ejercicio 2 · Nombres duplicados o con espacios en `aisles` y `departments`

In [69]:
resultados = []
for nombre, df, col in [('aisles', aisles, 'aisle'), ('departments', departments, 'department')]:
    resultados.append({
        'tabla': nombre,
        'nombres duplicados': int(df[col].duplicated().sum()),
        'espacios en los extremos': int((df[col] != df[col].str.strip()).sum()),
        'espacios dobles': int(df[col].str.contains(r'\s{2,}', regex=True).sum()),
        'espacio invisible': int(df[col].str.contains('\xa0', regex=False).sum()),
    })
res2 = pd.DataFrame(resultados)
display(res2)
necesita = res2.drop(columns='tabla').sum().sum() > 0
display(Markdown("**Interpretación:** " + ("hay problemas de texto que conviene estandarizar." if necesita else "no hay duplicados ni espacios sobrantes, así que `aisles` y `departments` **no necesitan limpieza de texto**.")))

,tabla,nombres duplicados,espacios en los extremos,espacios dobles,espacio invisible
0,aisles,0,0,0,0
1,departments,0,0,0,0


**Interpretación:** no hay duplicados ni espacios sobrantes, así que `aisles` y `departments` **no necesitan limpieza de texto**.

### Ejercicio 3 · Los nulos de `add_to_cart_order`

In [70]:
sin_posicion = order_prods[order_prods['add_to_cart_order'].isna()]
pedidos_con_nulos = sin_posicion['order_id'].unique()
nulos_por_pedido = sin_posicion.groupby('order_id').size()
tamano_pedidos = order_prods[order_prods['order_id'].isin(pedidos_con_nulos)].groupby('order_id').size()
max_posicion = int(order_prods['add_to_cart_order'].max())

print('Nulos:', len(sin_posicion))
print('Pedidos afectados:', len(pedidos_con_nulos))
print('Productos mínimos en esos pedidos:', int(tamano_pedidos.min()), '| máximo:', int(tamano_pedidos.max()))
print('Máxima posición registrada en toda la tabla:', max_posicion)
print('Nulos por pedido (mín, máx):', int(nulos_por_pedido.min()), int(nulos_por_pedido.max()))

if int(tamano_pedidos.min()) > max_posicion:
    causa = (f"todos los pedidos afectados tienen más de {max_posicion} productos y la posición máxima registrada es {max_posicion}: "
             f"el sistema dejó de numerar los productos después de la posición {max_posicion}. Es un **límite del registro**, no un error aleatorio.")
else:
    causa = "los pedidos afectados no se explican solo por su tamaño; habría que investigar otra causa con el área de datos."
display(Markdown(f"**Interpretación:** {len(sin_posicion)} nulos en {len(pedidos_con_nulos)} pedidos; cada uno tiene como mínimo {int(tamano_pedidos.min())} productos. Explicación: {causa}"))

Nulos: 836
Pedidos afectados: 70
Productos mínimos en esos pedidos: 65 | máximo: 127
Máxima posición registrada en toda la tabla: 64
Nulos por pedido (mín, máx): 1 63


**Interpretación:** 836 nulos en 70 pedidos; cada uno tiene como mínimo 65 productos. Explicación: todos los pedidos afectados tienen más de 64 productos y la posición máxima registrada es 64: el sistema dejó de numerar los productos después de la posición 64. Es un **límite del registro**, no un error aleatorio.

### Ejercicio 4 · Cuántos nombres cambiaron con la estandarización

In [71]:
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
tienen_nombre = products_crudo['product_name'].notna()
cambiaron = tienen_nombre & (products_crudo['product_name'] != products['product_name'])
print('Nombres no nulos:', int(tienen_nombre.sum()))
print('Nombres que cambiaron:', int(cambiaron.sum()))
ejemplos = pd.DataFrame({'antes': products_crudo.loc[cambiaron, 'product_name'],
                         'después': products.loc[cambiaron, 'product_name']}).head(5)
display(ejemplos)
display(Markdown(f"**Interpretación:** {fmt(cambiaron.sum())} de {fmt(tienen_nombre.sum())} nombres ({cambiaron.sum() / tienen_nombre.sum() * 100:.1f} %) cambiaron: espacios dobles, espacios invisibles y mayúsculas distintas. Sin este paso, el mismo producto podría contarse como varios."))

Nombres no nulos: 48436
Nombres que cambiaron: 6321


,antes,después
17,Pizza for One Suprema Frozen Pizza,Pizza For One Suprema Frozen Pizza
23,Tri-Vi-Sol® Vitamins A-C-and D Supplement Drop...,Tri-Vi-Sol® Vitamins A-C-And D Supplement Drop...
29,Three Cheese Ziti- Marinara with Meatballs,Three Cheese Ziti- Marinara With Meatballs
35,Traditional Lasagna with Meat Sauce Savory Ita...,Traditional Lasagna With Meat Sauce Savory Ita...
41,Biotin 1000 mcg,Biotin 1000 Mcg


**Interpretación:** 6.321 de 48.436 nombres (13.1 %) cambiaron: espacios dobles, espacios invisibles y mayúsculas distintas. Sin este paso, el mismo producto podría contarse como varios.

### Ejercicio 5 · Día de la semana como categoría ordenada

In [72]:
nombres_dia = {0: 'domingo', 1: 'lunes', 2: 'martes', 3: 'miércoles', 4: 'jueves', 5: 'viernes', 6: 'sábado'}
orden_dias = list(nombres_dia.values())

dia_texto = orders['order_dow'].map(nombres_dia)
dia_cat = pd.Series(pd.Categorical(dia_texto, categories=orden_dias, ordered=True), index=orders.index)

mem_texto = dia_texto.memory_usage(deep=True)
mem_cat = dia_cat.memory_usage(deep=True)
print('Memoria como texto:', round(mem_texto / 1e6, 2), 'MB')
print('Memoria como categoría:', round(mem_cat / 1e6, 2), 'MB')
print('Primer y último día del orden:', dia_cat.min(), '→', dia_cat.max())
print(dia_cat.value_counts().sort_index())
display(Markdown(f"**Interpretación:** la categoría ocupa {mem_cat / mem_texto * 100:.0f} % de la memoria del texto y además tiene un **orden** real (de domingo a sábado), por lo que `sort_values` y los gráficos respetan la semana y no el alfabeto."))

Memoria como texto: 7.14 MB
Memoria como categoría: 0.48 MB
Primer y último día del orden: domingo → sábado
domingo      84090
lunes        82185
martes       65833
miércoles    60897
jueves       59810
viernes      63488
sábado       62649
Name: count, dtype: int64


**Interpretación:** la categoría ocupa 7 % de la memoria del texto y además tiene un **orden** real (de domingo a sábado), por lo que `sort_values` y los gráficos respetan la semana y no el alfabeto.

### Ejercicio 6 · Filas de `order_prods` con productos `'Unknown'`

In [73]:
ids_unknown = products.loc[products['product_name'] == 'Unknown', 'product_id']
mascara = order_prods['product_id'].isin(ids_unknown)
filas_unknown = int(mascara.sum())
pedidos_unknown = order_prods.loc[mascara, 'order_id'].nunique()
print('Productos Unknown en el catálogo:', len(ids_unknown))
print('Filas de order_prods con productos Unknown:', filas_unknown)
print('Pedidos en que aparecen:', pedidos_unknown)
display(Markdown(
    f"**Interpretación:** {fmt(filas_unknown)} artículos ({filas_unknown / len(order_prods) * 100:.2f} % de `order_prods`) en {fmt(pedidos_unknown)} pedidos corresponden a productos sin nombre. "
    f"**No los eliminaría:** son compras reales; borrarlas distorsionaría el tamaño de los pedidos y las tasas de recompra. "
    f"Se conservan y, cuando se analicen productos por nombre, se excluyen solo en ese análisis."
))

Productos Unknown en el catálogo: 1258
Filas de order_prods con productos Unknown: 9547
Pedidos en que aparecen: 8254


**Interpretación:** 9.547 artículos (0.21 % de `order_prods`) en 8.254 pedidos corresponden a productos sin nombre. **No los eliminaría:** son compras reales; borrarlas distorsionaría el tamaño de los pedidos y las tasas de recompra. Se conservan y, cuando se analicen productos por nombre, se excluyen solo en ese análisis.

### Ejercicio 7 · Reemplazar `'missing'` por un nulo real en una copia de `departments`

In [74]:
departments_copia = departments.copy()
departments_copia['department'] = departments_copia['department'].replace('missing', pd.NA)
print(departments_copia.tail(3))

con_original = products.merge(departments, on='department_id', how='left').groupby('department').size()
con_copia = products.merge(departments_copia, on='department_id', how='left').groupby('department').size()
print('Productos contados por departamento (original):', fmt(con_original.sum()))
print('Productos contados por departamento (copia con NA):', fmt(con_copia.sum()))
display(Markdown(
    f"**Interpretación:** con el nulo real, `groupby` descarta en silencio {fmt(con_original.sum() - con_copia.sum())} productos porque ignora las claves nulas por defecto. "
    f"En la tabla original conviene mantener `'missing'` como una categoría visible para que esos artículos sigan apareciendo en las agrupaciones."
))

    department_id department
18             19     snacks
19             20       deli
20             21        NaN
Productos contados por departamento (original): 49.694
Productos contados por departamento (copia con NA): 48.436


**Interpretación:** con el nulo real, `groupby` descarta en silencio 1.258 productos porque ignora las claves nulas por defecto. En la tabla original conviene mantener `'missing'` como una categoría visible para que esos artículos sigan apareciendo en las agrupaciones.

---
# Lección 3 · Filtrado, ordenamiento y agrupación de datos

Cada pregunta se traduce a "*métrica* por *grupo*": "¿a qué hora se compra más?" es "número de pedidos por hora".

## 3.1 Ejemplos de la lección

### Filtrado con máscaras booleanas

In [75]:
es_primer_pedido = orders['order_number'] == 1
print(es_primer_pedido.head(3))
primeros = orders[es_primer_pedido]
print(len(primeros))
# Deberías ver tres valores booleanos y luego 28817

0    False
1    False
2    False
Name: order_number, dtype: bool
28817


In [76]:
es_fin_de_semana = orders['order_dow'].isin([0, 6])
es_manana = orders['order_hour_of_day'].between(6, 11)
print(len(orders[es_fin_de_semana & es_manana]))
# Deberías ver 45599

45599


### Filtrado con `query`

In [77]:
nocturnos_30 = orders.query('order_hour_of_day >= 22 and days_since_prior_order == 30')
print(len(nocturnos_30))
# Deberías ver 1449

1449


### Ordenamiento con `sort_values` y `nlargest`

In [78]:
print(orders.nlargest(3, 'order_number')[['user_id', 'order_number']])
historial = orders.sort_values(['user_id', 'order_number'])
print(historial[['user_id', 'order_number']].head(3))
# Deberías ver tres clientes con order_number 100; el cliente 2 aparece primero con sus pedidos 1 y 4

      user_id  order_number
2482     8220           100
4552   164906           100
6402   172294           100
        user_id  order_number
237012        2             1
317136        2             4
337239        4             2


### Agrupación con `groupby`

In [79]:
pedidos_por_hora = orders.groupby('order_hour_of_day')['order_id'].count()
print(pedidos_por_hora.idxmax(), pedidos_por_hora.max())
# Deberías ver 10 40578: la hora pico son las 10 a. m.

10 40578


In [80]:
resumen_dia = (orders
    .groupby('order_dow')
    .agg(pedidos=('order_id', 'count'),
         clientes=('user_id', 'nunique'),
         espera_media=('days_since_prior_order', 'mean'))
    .round(1))
resumen_dia

,pedidos,clientes,espera_media
order_dow,,,
0,84090,57661,11.8
1,82185,56479,11.3
2,65833,48587,11.2
3,60897,45331,10.7
4,59810,44281,10.5
5,63488,46127,10.5
6,62649,45604,11.4


### Tabla dinámica con `pivot_table`

In [81]:
matriz = orders.pivot_table(index='order_hour_of_day', columns='order_dow',
                            values='order_id', aggfunc='count')
print(matriz.shape)
matriz.loc[[8, 10, 14, 20]]
# Deberías ver (24, 7); la celda más alta es lunes a las 10, con 7881 pedidos

(24, 7)


order_dow,0,1,2,3,4,5,6
order_hour_of_day,,,,,,,
8,3908,4751,3485,3125,3136,3373,3246
10,6978,7881,5489,5026,4938,5347,4919
14,7533,6545,5344,4774,4795,5265,5375
20,2544,2332,2061,1917,2037,1886,1847


### Valores por fila con `transform`

In [82]:
orders['pedidos_cliente'] = orders.groupby('user_id')['order_id'].transform('count')
frecuentes = orders[orders['pedidos_cliente'] >= 5]
print(len(frecuentes), frecuentes['user_id'].nunique())
orders = orders.drop(columns='pedidos_cliente')
# Deberías ver 231946 30527

231946 30527


## 3.2 Laboratorio guiado 3 · Perfil de hábitos de compra

**Pregunta de negocio:** ¿a qué hora compra cada tipo de cliente? Plan: contar pedidos por cliente → segmentar clientes → clasificar pedidos por franja horaria → cruzar segmentos y franjas.

### Paso 1: pedidos por cliente

In [83]:
pedidos_por_cliente = orders.groupby('user_id')['order_id'].count()
print(len(pedidos_por_cliente))
pedidos_por_cliente.describe()
# Deberías ver 157437 clientes, mediana 2 y máximo 28

157437


count    157437.000000
mean          3.042182
std           2.746842
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max          28.000000
Name: order_id, dtype: float64

### Paso 2: segmentar clientes con `pd.cut`

**Ocasional** (1 pedido), **regular** (2 a 4) y **frecuente** (5 o más).

In [84]:
segmento_cliente = pd.cut(pedidos_por_cliente,
                          bins=[0, 1, 4, np.inf],
                          labels=['ocasional', 'regular', 'frecuente'])
print(segmento_cliente.value_counts().sort_index())
print((segmento_cliente.value_counts(normalize=True).sort_index() * 100).round(1))
# Deberías ver ocasional 55357 (35.2 %), regular 71553 (45.4 %), frecuente 30527 (19.4 %)

order_id
ocasional    55357
regular      71553
frecuente    30527
Name: count, dtype: int64
order_id
ocasional    35.2
regular      45.4
frecuente    19.4
Name: proportion, dtype: float64


### Paso 3: llevar el segmento a cada pedido con `map`

In [85]:
orders['segmento'] = orders['user_id'].map(segmento_cliente)
orders[['user_id', 'order_number', 'segmento']].head()

,user_id,order_number,segmento
0,183418,11,regular
1,163593,5,regular
2,39980,4,ocasional
3,82516,56,frecuente
4,196724,2,ocasional


### Paso 4: resumen por segmento

In [86]:
resumen_segmento = (orders
    .groupby('segmento', observed=True)
    .agg(clientes=('user_id', 'nunique'),
         pedidos=('order_id', 'count'),
         espera_mediana=('days_since_prior_order', 'median')))
resumen_segmento
# Deberías ver frecuente: 30527 clientes, 231946 pedidos y espera mediana 6 días

,clientes,pedidos,espera_mediana
segmento,,,
ocasional,55357,55357,16.0
regular,71553,191649,11.0
frecuente,30527,231946,6.0


In [87]:
frec = resumen_segmento.loc['frecuente']
pct_clientes = frec['clientes'] / resumen_segmento['clientes'].sum() * 100
pct_pedidos = frec['pedidos'] / resumen_segmento['pedidos'].sum() * 100
display(Markdown(
    f"**Interpretación:** el {pct_clientes:.1f} % de los clientes (los frecuentes) hace el {pct_pedidos:.1f} % de los pedidos y vuelve cada {frec['espera_mediana']:.0f} días (mediana). "
    f"Los ocasionales tienen una espera mayor ({resumen_segmento.loc['ocasional', 'espera_mediana']:.0f} días) y su único pedido en estos datos no es necesariamente el primero que hicieron en la tienda."
))

**Interpretación:** el 19.4 % de los clientes (los frecuentes) hace el 48.4 % de los pedidos y vuelve cada 6 días (mediana). Los ocasionales tienen una espera mayor (16 días) y su único pedido en estos datos no es necesariamente el primero que hicieron en la tienda.

### Paso 5: franjas horarias y tabla cruzada

In [88]:
orders['franja'] = pd.cut(orders['order_hour_of_day'],
                          bins=[-1, 5, 11, 17, 23],
                          labels=['madrugada', 'mañana', 'tarde', 'noche'])

perfil = (pd.crosstab(orders['segmento'], orders['franja'], normalize='index')
            .mul(100).round(1))
perfil
# Deberías ver ocasional: 2.1 / 30.0 / 49.4 / 18.5  ·  frecuente: 1.7 / 35.2 / 45.9 / 17.2

franja,madrugada,mañana,tarde,noche
segmento,,,,
ocasional,2.1,30.0,49.4,18.5
regular,1.9,31.6,48.4,18.1
frecuente,1.7,35.2,45.9,17.2


In [89]:
franja_top = perfil.idxmax(axis=1)
dif = perfil.loc['frecuente', 'mañana'] - perfil.loc['ocasional', 'mañana']
display(Markdown(
    f"**Interpretación:** todos los segmentos compran sobre todo en la franja **{franja_top.mode().iloc[0]}**, pero los frecuentes compran más en la mañana "
    f"({perfil.loc['frecuente', 'mañana']} % frente a {perfil.loc['ocasional', 'mañana']} % de los ocasionales; diferencia de {dif:.1f} puntos). "
    f"Para mercadeo, una campaña de fidelización debería llegar por la mañana."
))

**Interpretación:** todos los segmentos compran sobre todo en la franja **tarde**, pero los frecuentes compran más en la mañana (35.2 % frente a 30.0 % de los ocasionales; diferencia de 5.2 puntos). Para mercadeo, una campaña de fidelización debería llegar por la mañana.

### Paso 6: las combinaciones de día y franja más activas

In [90]:
(orders.groupby(['order_dow', 'franja'], observed=True)
       .size()
       .sort_values(ascending=False)
       .head(3))
# Deberías ver domingo-tarde (42340), lunes-tarde (37432) y martes-tarde (30751)

order_dow  franja
0          tarde     42340
1          tarde     37432
2          tarde     30751
dtype: int64

### Paso 7: dejar `orders` como estaba

In [91]:
orders = orders.drop(columns=['segmento', 'franja'], errors='ignore')
print(orders.shape)
# Deberías ver (478952, 6)

(478952, 6)


## 3.3 Reto 3 · Siete ejercicios

### Ejercicio 1 · Las tres horas con más pedidos los domingos
*Métrica:* pedidos · *grupo:* hora · *condición:* `order_dow == 0`.

In [92]:
domingos = orders[orders['order_dow'] == 0]
top_horas_domingo = domingos.groupby('order_hour_of_day')['order_id'].count().nlargest(3)
print(top_horas_domingo)
h = top_horas_domingo
display(Markdown(f"**Interpretación:** los domingos las horas pico son las {h.index[0]} h ({fmt(h.iloc[0])} pedidos), las {h.index[1]} h ({fmt(h.iloc[1])}) y las {h.index[2]} h ({fmt(h.iloc[2])}): concentran la actividad de media mañana."))

order_hour_of_day
15    7585
14    7533
13    7483
Name: order_id, dtype: int64


**Interpretación:** los domingos las horas pico son las 15 h (7.585 pedidos), las 14 h (7.533) y las 13 h (7.483): concentran la actividad de media mañana.

### Ejercicio 2 · Hora con mayor y menor espera media entre pedidos

In [93]:
espera_por_hora = orders.groupby('order_hour_of_day')['days_since_prior_order'].mean().astype('float64')
hora_max, hora_min = espera_por_hora.idxmax(), espera_por_hora.idxmin()
print('Mayor espera:', hora_max, 'h →', round(espera_por_hora.max(), 2), 'días')
print('Menor espera:', hora_min, 'h →', round(espera_por_hora.min(), 2), 'días')
display(Markdown(f"**Interpretación:** quienes compran a las {hora_max} h tardan más en volver ({espera_por_hora.max():.1f} días en promedio) y quienes compran a las {hora_min} h vuelven antes ({espera_por_hora.min():.1f} días). La diferencia es de {espera_por_hora.max() - espera_por_hora.min():.1f} días."))

Mayor espera: 0 h → 12.52 días
Menor espera: 22 h → 10.62 días


**Interpretación:** quienes compran a las 0 h tardan más en volver (12.5 días en promedio) y quienes compran a las 22 h vuelven antes (10.6 días). La diferencia es de 1.9 días.

### Ejercicio 3 · Clientes con 10 pedidos o más

In [94]:
clientes_10 = int((pedidos_por_cliente >= 10).sum())
print(clientes_10, 'clientes con 10 pedidos o más de', fmt(len(pedidos_por_cliente)))
display(Markdown(f"**Interpretación:** {fmt(clientes_10)} clientes ({clientes_10 / len(pedidos_por_cliente) * 100:.1f} %) tienen 10 pedidos o más en este archivo: son el grupo más fiel."))

6327 clientes con 10 pedidos o más de 157.437


**Interpretación:** 6.327 clientes (4.0 %) tienen 10 pedidos o más en este archivo: son el grupo más fiel.

### Ejercicio 4 · Pedidos hechos exactamente 7 días después del anterior

In [95]:
exactos_7 = int((orders['days_since_prior_order'] == 7).sum())
pct_7 = exactos_7 / len(orders) * 100
print(fmt(exactos_7), 'pedidos →', round(pct_7, 2), '% del total')
display(Markdown(f"**Interpretación:** {fmt(exactos_7)} pedidos ({pct_7:.1f} % del total) se hicieron exactamente una semana después del anterior, lo que sugiere un hábito de compra semanal."))

44.577 pedidos → 9.31 % del total


**Interpretación:** 44.577 pedidos (9.3 % del total) se hicieron exactamente una semana después del anterior, lo que sugiere un hábito de compra semanal.

### Ejercicio 5 · Pedidos con 20 productos o más y el pedido más grande (`order_prods`)

In [96]:
productos_por_pedido = order_prods.groupby('order_id').size()
pedidos_20 = int((productos_por_pedido >= 20).sum())
pedido_max = productos_por_pedido.idxmax()
print('Pedidos con 20 productos o más:', fmt(pedidos_20))
print('Pedido más grande:', pedido_max, 'con', productos_por_pedido.max(), 'productos')
display(Markdown(f"**Interpretación:** {fmt(pedidos_20)} pedidos ({pedidos_20 / len(productos_por_pedido) * 100:.1f} % de los que tienen productos) llegan a 20 productos o más. El mayor es el pedido {pedido_max}, con {productos_por_pedido.max()} productos."))

Pedidos con 20 productos o más: 49.420
Pedido más grande: 61355 con 127 productos


**Interpretación:** 49.420 pedidos (11.0 % de los que tienen productos) llegan a 20 productos o más. El mayor es el pedido 61355, con 127 productos.

### Ejercicio 6 · Tasa de recompra por posición del carrito (posiciones 1 a 5)

In [97]:
primeras = order_prods[order_prods['add_to_cart_order'].between(1, 5)]
tasa_pos = primeras.groupby('add_to_cart_order')['reordered'].mean().round(3)
print(tasa_pos)
baja = tasa_pos.is_monotonic_decreasing
display(Markdown(
    f"**Interpretación:** la tasa de recompra va de {tasa_pos.iloc[0]:.3f} en la posición 1 a {tasa_pos.iloc[-1]:.3f} en la posición 5. "
    + ("Baja de forma constante: los primeros productos que el cliente agrega suelen ser los que ya conoce y compra siempre; los últimos son más exploratorios." if baja else "No es una bajada constante, así que la posición por sí sola no explica la recompra.")
))

add_to_cart_order
1    0.679
2    0.677
3    0.659
4    0.638
5    0.617
Name: reordered, dtype: float64


**Interpretación:** la tasa de recompra va de 0.679 en la posición 1 a 0.617 en la posición 5. Baja de forma constante: los primeros productos que el cliente agrega suelen ser los que ya conoce y compra siempre; los últimos son más exploratorios.

### Ejercicio 7 · Mediana de espera: fin de semana frente al resto de la semana

In [98]:
mediana_fds = orders.groupby(orders['order_dow'].isin([0, 6]))['days_since_prior_order'].median()
print(mediana_fds.rename({True: 'fin de semana', False: 'resto de la semana'}))
display(Markdown(f"**Interpretación:** la mediana de espera es de {mediana_fds[True]:.0f} días en los pedidos de fin de semana y de {mediana_fds[False]:.0f} días en el resto de la semana."))

order_dow
resto de la semana    7.0
fin de semana         8.0
Name: days_since_prior_order, dtype: Float64


**Interpretación:** la mediana de espera es de 8 días en los pedidos de fin de semana y de 7 días en el resto de la semana.

---
# Lección 4 · Combinación y estructuración de múltiples datasets

`merge` une **columnas** emparejando claves; `concat` apila **filas**. Un `merge` mal hecho no da error: multiplica o pierde filas en silencio, por eso cada unión se comprueba (`validate`, `indicator` y el tamaño de la tabla).

## 4.1 Ejemplos de la lección

### Tipos de unión con `how`

In [99]:
pedidos = pd.DataFrame({'order_id': [1, 2, 3], 'user_id': [10, 20, 30]})
detalle = pd.DataFrame({'order_id': [1, 1, 2, 4], 'product_id': [100, 200, 100, 300]})

for tipo in ['inner', 'left', 'outer']:
    resultado = pedidos.merge(detalle, on='order_id', how=tipo)
    print(tipo, len(resultado))
    print(resultado, '\n')
# Deberías ver inner 3, left 4 y outer 5

inner 3
   order_id  user_id  product_id
0         1       10         100
1         1       10         200
2         2       20         100 

left 4
   order_id  user_id  product_id
0         1       10       100.0
1         1       10       200.0
2         2       20       100.0
3         3       30         NaN 

outer 5
   order_id  user_id  product_id
0         1     10.0       100.0
1         1     10.0       200.0
2         2     20.0       100.0
3         3     30.0         NaN
4         4      NaN       300.0 



### Validación de la relación con `validate`

In [100]:
catalogo_malo = pd.concat([products, products.head(1)])

try:
    order_prods.merge(catalogo_malo, on='product_id', validate='m:1')
except pd.errors.MergeError as error:
    print('Detectado:', str(error).splitlines()[0])
# Deberías ver: Merge keys are not unique in right dataset; not a many-to-one merge

Detectado: Merge keys are not unique in right dataset; not a many-to-one merge


### Auditoría de la unión con `indicator`

In [101]:
auditoria = orders.merge(order_prods[['order_id']].drop_duplicates(),
                         on='order_id', how='left', indicator=True)
conteo_auditoria = auditoria['_merge'].value_counts()
conteo_auditoria
# Deberías ver both 450046, left_only 28906 y right_only 0

_merge
both          450046
left_only      28906
right_only         0
Name: count, dtype: int64

In [102]:
sin_productos = int(conteo_auditoria['left_only'])
display(Markdown(f"**Interpretación:** {fmt(sin_productos)} pedidos ({sin_productos / len(orders) * 100:.1f} %) no tienen productos registrados. Con un `inner` habrían desaparecido sin aviso."))

**Interpretación:** 28.906 pedidos (6.0 %) no tienen productos registrados. Con un `inner` habrían desaparecido sin aviso.

### Construcción de la tabla maestra

Se parte de la **tabla de hechos** (`order_prods`, la más detallada) y se añaden las dimensiones, comprobando el tamaño en cada paso.

In [103]:
filas_inicio = len(order_prods)

maestra = order_prods.merge(products, on='product_id', how='left', validate='m:1')
print('+ products   ', maestra.shape)
maestra = maestra.merge(aisles, on='aisle_id', how='left', validate='m:1')
print('+ aisles     ', maestra.shape)
maestra = maestra.merge(departments, on='department_id', how='left', validate='m:1')
print('+ departments', maestra.shape)
maestra = maestra.merge(orders, on='order_id', how='left', validate='m:1')
print('+ orders     ', maestra.shape)

assert len(maestra) == filas_inicio
# Deberías ver siempre 4545007 filas, con 7, 8, 9 y finalmente 14 columnas

+ products    (4545007, 7)
+ aisles      (4545007, 8)
+ departments (4545007, 9)
+ orders      (4545007, 14)


### Preguntas sobre la tabla maestra

In [104]:
top = (maestra.groupby(['product_id', 'product_name']).size()
              .nlargest(5).reset_index(name='veces'))
display(top)

por_depto = (maestra.groupby('department')
                    .agg(articulos=('product_id', 'size'),
                         tasa_recompra=('reordered', 'mean'))
                    .sort_values('articulos', ascending=False))
por_depto.head(3)
# Deberías ver Banana (66050) primero; produce 1328302 artículos (0.651), dairy eggs 759842 (0.671), snacks 404850 (0.574)

,product_id,product_name,veces
0,24852,Banana,66050
1,13176,Bag Of Organic Bananas,53297
2,21137,Organic Strawberries,37039
3,21903,Organic Baby Spinach,33971
4,47209,Organic Hass Avocado,29773


,articulos,tasa_recompra
department,,
produce,1328302,0.651140
dairy eggs,759842,0.670597
snacks,404850,0.574442


### Apilado con `concat`

In [105]:
categorias = pd.concat([
    aisles.rename(columns={'aisle_id': 'id', 'aisle': 'nombre'}).assign(tipo='pasillo'),
    departments.rename(columns={'department_id': 'id', 'department': 'nombre'}).assign(tipo='departamento'),
], ignore_index=True)
print(categorias.shape)
# Deberías ver (155, 3)

(155, 3)


### Formato ancho y largo con `pivot_table` y `melt`

In [106]:
ancho = orders.pivot_table(index='order_hour_of_day', columns='order_dow',
                           values='order_id', aggfunc='count')
largo = (ancho.reset_index()
              .melt(id_vars='order_hour_of_day', var_name='order_dow', value_name='pedidos'))
print(ancho.shape, largo.shape)
# Deberías ver (24, 7) (168, 3)

(24, 7) (168, 3)


### Guardado del resultado

Parquet conserva los tipos y ocupa menos (requiere `pyarrow`); CSV se abre en Excel. Todo va a `data/processed/`.

In [107]:
maestra.to_parquet(SALIDA / 'instacart_maestra.parquet', index=False)
por_depto.to_csv(SALIDA / 'resumen_departamentos.csv')
print(sorted(p.name for p in SALIDA.iterdir()))

['instacart_maestra.parquet', 'resumen_departamentos.csv']


## 4.2 Laboratorio guiado 4 · Tabla de análisis por producto

**Pregunta de compras:** ¿cuál es el producto más fiel de cada departamento? Patrón: **agregar** la tabla maestra a otro nivel (de artículo a producto) → **volver a unir** las descripciones → **clasificar dentro de cada grupo**.

### Paso 1: agregar la tabla maestra por producto

In [108]:
por_producto = (maestra
    .groupby('product_id')
    .agg(veces=('order_id', 'size'),
         clientes=('user_id', 'nunique'),
         tasa_recompra=('reordered', 'mean'),
         posicion_media=('add_to_cart_order', 'mean'))
    .reset_index())
print(por_producto.shape)
por_producto.head(3)
# Deberías ver (45573, 5); el producto 1 se pidió 280 veces por 206 clientes, con tasa de recompra 0.564

(45573, 5)


,product_id,veces,clientes,tasa_recompra,posicion_media
0,1,280,206,0.564286,5.642857
1,2,11,11,0.000000,8.727273
2,3,42,27,0.738095,6.952381


### Paso 2: recuperar nombre y departamento con `merge`

In [109]:
ficha_productos = (por_producto
    .merge(products, on='product_id', how='left', validate='1:1')
    .merge(departments, on='department_id', how='left', validate='m:1')
    [['product_id', 'product_name', 'department', 'veces', 'clientes',
      'tasa_recompra', 'posicion_media']])
print(ficha_productos.shape)
ficha_productos.loc[ficha_productos['veces'].idxmax()]
# Deberías ver (45573, 7) y la fila de Banana: produce, 66050 veces, 33274 clientes, recompra 0.844, posición media 4.87

(45573, 7)


product_id           24852
product_name        Banana
department         produce
veces                66050
clientes             33274
tasa_recompra     0.844254
posicion_media    4.874815
Name: 22808, dtype: object

### Paso 3: filtrar los productos con suficientes datos

Una tasa de recompra calculada con 3 pedidos no es confiable: se exigen al menos 1 000 pedidos.

In [110]:
populares = ficha_productos[ficha_productos['veces'] >= 1000].copy()
print(len(populares))
# Deberías ver 739

739


### Paso 4: clasificar dentro de cada departamento con `rank`

In [111]:
populares['puesto'] = (populares
    .groupby('department')['tasa_recompra']
    .rank(ascending=False, method='first'))

lideres = (populares[populares['puesto'] == 1]
           .sort_values('tasa_recompra', ascending=False))
print(len(lideres))
lideres[['department', 'product_name', 'veces', 'tasa_recompra']].head(5)
# Deberías ver 18 líderes; primero dairy eggs (Whole Organic Omega 3 Milk, 0.863) y luego produce (Banana, 0.844)

18


,department,product_name,veces,tasa_recompra
41704,dairy eggs,Whole Organic Omega 3 Milk,1286,0.863142
22808,produce,Banana,66050,0.844254
29030,snacks,Extra Fancy Unsalted Mixed Nuts,1436,0.822423
42302,beverages,Zero Calorie Cola,1256,0.818471
3570,bakery,Organic Honey Sweet Whole Wheat Bread,1607,0.799004


In [112]:
mas_pedido = ficha_productos.loc[ficha_productos['veces'].idxmax()]
lider_lacteos = lideres[lideres['department'] == 'dairy eggs'].iloc[0]
display(Markdown(
    f"**Interpretación:** el producto más pedido ({mas_pedido['product_name']}, {fmt(mas_pedido['veces'])} veces) no siempre es el más fiel. "
    f"En lácteos, **{lider_lacteos['product_name']}** tiene {fmt(lider_lacteos['veces'])} pedidos pero una tasa de recompra de {lider_lacteos['tasa_recompra']:.3f}: quien la compra casi siempre la repite."
))

**Interpretación:** el producto más pedido (Banana, 66.050 veces) no siempre es el más fiel. En lácteos, **Whole Organic Omega 3 Milk** tiene 1.286 pedidos pero una tasa de recompra de 0.863: quien la compra casi siempre la repite.

### Paso 5: la misma respuesta con `sort_values` y `head`

In [113]:
lideres_2 = (populares
    .sort_values('tasa_recompra', ascending=False)
    .groupby('department')
    .head(1))
print(len(lideres_2), set(lideres_2['product_id']) == set(lideres['product_id']))
# Deberías ver 18 True

18 True


### Paso 6: guardar la ficha de productos

In [114]:
ficha_productos.to_parquet(SALIDA / 'ficha_productos.parquet', index=False)
lideres.to_csv(SALIDA / 'lideres_por_departamento.csv', index=False)
print(sorted(p.name for p in SALIDA.iterdir()))

['ficha_productos.parquet', 'instacart_maestra.parquet', 'lideres_por_departamento.csv', 'resumen_departamentos.csv']


## 4.3 Reto 4 · Siete ejercicios (con la tabla `maestra`)

### Ejercicio 1 · Pasillos con mayor y menor tasa de recompra

In [115]:
por_pasillo = (maestra.groupby('aisle')
                      .agg(articulos=('product_id', 'size'), tasa_recompra=('reordered', 'mean')))
grandes = por_pasillo[por_pasillo['articulos'] >= 10000]
top3_pasillos = grandes.nlargest(3, 'tasa_recompra').round(3)
pasillo_min = por_pasillo['tasa_recompra'].idxmin()
print('Pasillos con al menos 10.000 artículos:', len(grandes))
display(top3_pasillos)
print('Tasa más baja de todos los pasillos:', pasillo_min, round(por_pasillo.loc[pasillo_min, 'tasa_recompra'], 3),
      f"({fmt(por_pasillo.loc[pasillo_min, 'articulos'])} artículos)")
display(Markdown(
    f"**Interpretación:** entre los pasillos grandes, los de mayor recompra son {', '.join(top3_pasillos.index)} (productos de consumo habitual). "
    f"El de menor recompra de todos es **{pasillo_min}** ({por_pasillo.loc[pasillo_min, 'tasa_recompra']:.3f}), con {fmt(por_pasillo.loc[pasillo_min, 'articulos'])} artículos."
))

Pasillos con al menos 10.000 artículos: 77


,articulos,tasa_recompra
aisle,,
milk,125522,0.781
water seltzer sparkling water,117133,0.731
fresh fruits,510101,0.718


Tasa más baja de todos los pasillos: spices seasonings 0.152 (29.454 artículos)


**Interpretación:** entre los pasillos grandes, los de mayor recompra son milk, water seltzer sparkling water, fresh fruits (productos de consumo habitual). El de menor recompra de todos es **spices seasonings** (0.152), con 29.454 artículos.

### Ejercicio 2 · Participación por departamento en la madrugada (0 a 5 h) frente al total

In [116]:
madrugada = maestra.loc[maestra['order_hour_of_day'].between(0, 5), 'department']
part_madrugada = madrugada.value_counts(normalize=True) * 100
part_total = maestra['department'].value_counts(normalize=True) * 100
comparacion = pd.DataFrame({'madrugada_%': part_madrugada, 'total_%': part_total}).round(2)
comparacion['diferencia_puntos'] = (comparacion['madrugada_%'] - comparacion['total_%']).round(2)
display(comparacion.sort_values('madrugada_%', ascending=False).head(6))

top3_mad = set(part_madrugada.nlargest(3).index)
top3_tot = set(part_total.nlargest(3).index)
solo_madrugada = sorted(top3_mad - top3_tot)
print('Top 3 madrugada:', sorted(top3_mad))
print('Top 3 total:', sorted(top3_tot))
print('Entra en la madrugada pero no en el total:', solo_madrugada)
display(Markdown(
    "**Interpretación:** " + (f"el departamento **{', '.join(solo_madrugada)}** está en el top 3 de la madrugada pero no en el top 3 total: de madrugada pesa más que en el resto del día."
                              if solo_madrugada else "el top 3 de la madrugada coincide con el total; solo cambian los porcentajes.")
))

,madrugada_%,total_%,diferencia_puntos
department,,,
produce,30.21,29.23,0.98
dairy eggs,15.87,16.72,-0.85
beverages,8.49,8.26,0.23
snacks,7.89,8.91,-1.02
frozen,7.65,6.91,0.74
pantry,6.17,5.78,0.39


Top 3 madrugada: ['beverages', 'dairy eggs', 'produce']
Top 3 total: ['dairy eggs', 'produce', 'snacks']
Entra en la madrugada pero no en el total: ['beverages']


**Interpretación:** el departamento **beverages** está en el top 3 de la madrugada pero no en el top 3 total: de madrugada pesa más que en el resto del día.

### Ejercicio 3 · Los tres productos más pedidos del departamento `snacks`

In [117]:
top3_snacks = (maestra[maestra['department'] == 'snacks']
               .groupby(['product_id', 'product_name']).size()
               .nlargest(3).reset_index(name='veces'))
display(top3_snacks)
display(Markdown(f"**Interpretación:** los tres snacks más pedidos son {', '.join(top3_snacks['product_name'])}; el primero se pidió {fmt(top3_snacks['veces'].iloc[0])} veces."))

,product_id,product_name,veces
0,13870,Lightly Salted Baked Snap Pea Crisps,3462
1,34448,Original Veggie Straws,2331
2,31040,Sea Salt Pita Chips,2275


**Interpretación:** los tres snacks más pedidos son Lightly Salted Baked Snap Pea Crisps, Original Veggie Straws, Sea Salt Pita Chips; el primero se pidió 3.462 veces.

### Ejercicio 4 · Tabla con una fila por cliente

In [118]:
por_cliente = (maestra.groupby('user_id')
                      .agg(pedidos=('order_id', 'nunique'),
                           articulos=('product_id', 'size'),
                           tasa_recompra=('reordered', 'mean')))
cliente_top = por_cliente['articulos'].idxmax()
print('Clientes:', fmt(len(por_cliente)))
print('Promedio de artículos por cliente:', round(por_cliente['articulos'].mean(), 1))
print('Cliente que compró más artículos:', cliente_top, '→', por_cliente.loc[cliente_top, 'articulos'], 'artículos')
display(por_cliente.head(3))
display(Markdown(
    f"**Interpretación:** la tabla tiene {fmt(len(por_cliente))} clientes con productos registrados (no los {fmt(orders['user_id'].nunique())} de `orders`, porque hay pedidos sin productos). "
    f"En promedio cada cliente compró {por_cliente['articulos'].mean():.1f} artículos; el mayor es el cliente {cliente_top}, con {por_cliente.loc[cliente_top, 'articulos']}."
))

Clientes: 149.626
Promedio de artículos por cliente: 30.4
Cliente que compró más artículos: 103634 → 580 artículos


,pedidos,articulos,tasa_recompra
user_id,,,
2,2,26,0.038462
4,1,2,0.000000
5,1,12,0.666667


**Interpretación:** la tabla tiene 149.626 clientes con productos registrados (no los 157.437 de `orders`, porque hay pedidos sin productos). En promedio cada cliente compró 30.4 artículos; el mayor es el cliente 103634, con 580.

### Ejercicio 5 · `orders` con `order_prods`: `inner` frente a `left`

In [119]:
n_inner = len(orders.merge(order_prods, on='order_id', how='inner', validate='1:m'))
n_left = len(orders.merge(order_prods, on='order_id', how='left', validate='1:m'))
print('inner:', fmt(n_inner), '| left:', fmt(n_left), '| diferencia:', fmt(n_left - n_inner))
display(Markdown(
    f"**Interpretación:** `inner` devuelve {fmt(n_inner)} filas (una por artículo) y `left` {fmt(n_left)}. La diferencia de {fmt(n_left - n_inner)} son los pedidos de `orders` **sin productos**: "
    f"`left` los conserva con nulos (una fila cada uno) y `inner` los descarta."
))

inner: 4.545.007 | left: 4.573.913 | diferencia: 28.906


**Interpretación:** `inner` devuelve 4.545.007 filas (una por artículo) y `left` 4.573.913. La diferencia de 28.906 son los pedidos de `orders` **sin productos**: `left` los conserva con nulos (una fila cada uno) y `inner` los descarta.

### Ejercicio 6 · `concat` de los tres productos más pedidos de `produce` y `dairy eggs`

In [120]:
partes = []
for depto in ['produce', 'dairy eggs']:
    parte = (ficha_productos[ficha_productos['department'] == depto]
             .nlargest(3, 'veces')[['product_id', 'product_name', 'veces']]
             .assign(departamento=depto))
    partes.append(parte)
top_dos_deptos = pd.concat(partes, ignore_index=True)
display(top_dos_deptos)
display(Markdown("**Interpretación:** `concat` apila las dos tablas de tres filas y la columna `departamento` permite saber de dónde viene cada producto; sin ella no se podrían distinguir."))

,product_id,product_name,veces,departamento
0,24852,Banana,66050,produce
1,13176,Bag Of Organic Bananas,53297,produce
2,21137,Organic Strawberries,37039,produce
3,27845,Organic Whole Milk,19600,dairy eggs
4,49235,Organic Half & Half,10752,dairy eggs
5,27086,Half & Half,9588,dairy eggs


**Interpretación:** `concat` apila las dos tablas de tres filas y la columna `departamento` permite saber de dónde viene cada producto; sin ella no se podrían distinguir.

### Ejercicio 7 · Tabla dinámica de artículos por departamento y día de la semana

In [121]:
dias = {0: 'domingo', 1: 'lunes', 2: 'martes', 3: 'miércoles', 4: 'jueves', 5: 'viernes', 6: 'sábado'}
dinamica = (maestra.pivot_table(index='department', columns='order_dow',
                                values='product_id', aggfunc='count')
                   .rename(columns=dias))
print('Forma:', dinamica.shape)
dia_alcohol = dinamica.loc['alcohol'].idxmax()
display(dinamica.loc[['alcohol', 'produce']])
display(Markdown(
    f"**Interpretación:** la tabla tiene forma {dinamica.shape} (departamentos × días). El día con más artículos de `alcohol` es **{dia_alcohol}** "
    f"({fmt(dinamica.loc['alcohol', dia_alcohol])} artículos)."
))

Forma: (21, 7)


order_dow,domingo,lunes,martes,miércoles,jueves,viernes,sábado
department,,,,,,,
alcohol,2397,3025,2868,3094,3656,3814,2787
produce,272745,236672,167248,149120,148340,166068,188109


**Interpretación:** la tabla tiene forma (21, 7) (departamentos × días). El día con más artículos de `alcohol` es **viernes** (3.814 artículos).

---
## Informe de procesamiento

Esta celda genera `reports/processing_report.md` a partir de los resultados reales de este cuaderno (cifras, tablas y archivos guardados).

In [122]:
def md_tabla(df, indice=True):
    d = df.reset_index() if indice else df.copy()
    cols = [str(c) for c in d.columns]
    filas = ['| ' + ' | '.join(cols) + ' |', '| ' + ' | '.join('---' for _ in cols) + ' |']
    for fila in d.itertuples(index=False):
        filas.append('| ' + ' | '.join(str(v) for v in fila) + ' |')
    return '\n'.join(filas)

archivos = []
for p in sorted(SALIDA.iterdir()):
    if p.is_file():
        archivos.append({'archivo': p.name, 'tamaño_MB': round(p.stat().st_size / 1e6, 2)})
tabla_archivos = pd.DataFrame(archivos)

L = []
L.append('# Informe de procesamiento · Instacart')
L.append('')
L.append('Actividad GA1-220501094-01-AA1-EV06 · Limpieza, análisis y estructuración de datos con Pandas. Este informe se genera desde `notebooks/pandas_labs.ipynb`.')
L.append('')
L.append('## 1. Resumen')
L.append('')
L.append(f'- `orders`: {fmt(len(orders))} pedidos de {fmt(orders["user_id"].nunique())} clientes; `products`: {fmt(len(products))} productos en {len(aisles)} pasillos y {len(departments)} departamentos; `order_prods`: {fmt(len(order_prods))} artículos.')
L.append(f'- La tabla maestra combina las cinco tablas: {fmt(maestra.shape[0])} filas y {maestra.shape[1]} columnas, sin perder ni duplicar filas (`validate` y `assert`).')
L.append('')
L.append('## 2. Lección 2 · Limpieza')
L.append('')
L.append('### Informe de cambios del pipeline')
L.append('')
L.append(md_tabla(informe, indice=False))
L.append('')
L.append('### Decisiones')
L.append('')
L.append('| Problema | Evidencia | Decisión |')
L.append('| --- | --- | --- |')
L.append(f'| Pedidos duplicados | {dup_quitados} copias exactas, todas el mismo día y hora | Eliminar |')
L.append('| Tipos de dato | Enteros de 64 bits y nulos en columnas enteras | Reducir con `astype` y usar `Int16` (nullable) |')
L.append(f'| `days_since_prior_order` nulo | Cada nulo es un primer pedido | Conservar el nulo |')
L.append(f'| `product_name` nulo | {nulos_pr0} productos del pasillo y departamento `missing` | Rellenar con `Unknown` |')
L.append(f'| `add_to_cart_order` nulo | {nulos_op} nulos (Reto 2, ejercicio 3) | Conservar el nulo |')
L.append('| Nombres con espacios y mayúsculas | Espacios dobles y espacios invisibles | Estandarizar con `.str` |')
L.append('| Duplicados implícitos | Mismo nombre con `product_id` distintos | Conservar y agrupar por `product_id` |')
L.append('')
L.append('## 3. Lección 3 · Análisis')
L.append('')
L.append(f'- Hora pico de pedidos: las {pedidos_por_hora.idxmax()} h ({fmt(pedidos_por_hora.max())} pedidos).')
L.append(f'- Clientes por segmento (según su número de pedidos): ver tabla.')
L.append('')
L.append(md_tabla(resumen_segmento))
L.append('')
L.append('Perfil por franja horaria (% de pedidos de cada segmento):')
L.append('')
L.append(md_tabla(perfil))
L.append('')
L.append('## 4. Lección 4 · Combinación')
L.append('')
L.append(f'- Auditoría `orders` ↔ `order_prods`: {fmt(conteo_auditoria["both"])} pedidos con productos y {fmt(conteo_auditoria["left_only"])} sin productos registrados.')
L.append(f'- Tabla por producto: {fmt(len(ficha_productos))} productos; se analizaron los {len(populares)} con al menos 1.000 pedidos.')
L.append('')
L.append('Producto más fiel de cada departamento (5 primeros):')
L.append('')
L.append(md_tabla(lideres[['department', 'product_name', 'veces', 'tasa_recompra']].head(5).round(3), indice=False))
L.append('')
L.append('## 5. Archivos generados en `data/processed/`')
L.append('')
L.append(md_tabla(tabla_archivos, indice=False))
L.append('')
L.append('## 6. Limitaciones')
L.append('')
L.append('- Los datos no traen fechas calendario ni precios; solo día de la semana y hora.')
L.append(f'- {fmt(conteo_auditoria["left_only"])} pedidos no tienen productos registrados.')
L.append(f'- `add_to_cart_order` tiene {nulos_op} nulos que se conservaron.')
L.append('- Los productos `Unknown` se conservan; hay que excluirlos al analizar por nombre.')
L.append('- `instacart_maestra.parquet` es grande y no se sube a GitHub; se regenera ejecutando el cuaderno.')

ruta_informe = INFORMES / 'processing_report.md'
ruta_informe.write_text('\n'.join(L), encoding='utf-8')
print('Informe guardado en', ruta_informe.resolve())

Informe guardado en C:\Users\IK\Documents\instacart-analisis\reports\processing_report.md


---
## Verificación final

**Restart** y luego **Run All**: todas las celdas deben ejecutarse sin error, y deben existir los archivos de `data/processed/` y `reports/processing_report.md`.